# Replication Code for "Predicting Human Mobility Using Dense Smartphone GPS Trajectories and Transformer Models"

## Disclaimer on Exact Reproducibility Across GPU Hardware

Even when every source of randomness is fully seeded (Python random, NumPy, PyTorch, data‐loader workers), and the exact same library binaries (PyTorch 2.4.1+cu121, CUDA 12.1, cuDNN 9.1, NumPy 1.23.5, etc.) are installed, bit‐for‐bit identical results can only be guaranteed on the same GPU architecture. For example, our primary experiments were run on an NVIDIA RTX A5000 (Driver 560.35.05, CUDA 12.6), and the deterministic cuDNN kernels selected on that card produce a very specific floating‐point rounding path. Colab typically provides T4, P100, or V100 GPUs, which—even under a “deterministic” build of cuDNN—invoke different optimized kernels and may accumulate minute floating‐point differences over hundreds of weight updates. As a result, anyone who is running the same code on Colab GPUs should expect functionally equivalent behavior (identical losses up to ≈1e-6), but they will not see precisely the same final weights or epoch‐by‐epoch outputs unless they use an RTX A5000 (or another card with identical compute capability and driver).

# Setting up environment

In [ ]:
# Uninstall pre-installed torch, torchvision, torchaudio, numpy
!pip uninstall -y torch torchvision torchaudio numpy

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2


In [ ]:
# Install torch 2.4.1+cu121, torchvision 0.19.1+cu121 and torchaudio 2.4.1+cu121
!pip install \
    torch==2.4.1+cu121 \
    torchvision==0.19.1+cu121 \
    torchaudio==2.4.1+cu121 \
    --extra-index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 88.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 121.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 115.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 138.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 43.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import numpy as np
print("Colab PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "cuDNN:", torch.backends.cudnn.version())
print("NumPy:", np.__version__)

Colab PyTorch: 2.4.1+cu121 CUDA: 12.1 cuDNN: 90100
NumPy: 2.0.2


# Downloading Repo and Data

This section downloads the published codebase (_SpeedTransformer_) and the three pre‑processed datasets (_MOBIS_, _GeoLife_, and _Miniprogram_) exactly as referenced in **Section “Data and Codes Availability”**.  
The repository zip is fetched from Zenodo and unzipped into the Colab working directory.  
The subsequent cell creates a `data/` folder where the CSV files are stored so that the training scripts can locate them via the relative paths used throughout the notebook.

In [ ]:
!wget /content/SpeedTransformer.zip https://zenodo.org/records/17290562/files/SpeedTransformer.zip
!unzip /content/SpeedTransformer.zip

/content/SpeedTransformer.zip: Scheme missing.
--2025-10-08 14:39:10--  https://zenodo.org/records/17290562/files/SpeedTransformer.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.25, 188.185.48.194, 188.185.45.92, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.25|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67645 (66K) [application/octet-stream]
Saving to: ‘SpeedTransformer.zip’

SpeedTransformer.zi 100%[===================>]  66.06K  --.-KB/s    in 0.06s   

2025-10-08 14:39:10 (1.11 MB/s) - ‘SpeedTransformer.zip’ saved [67645/67645]

FINISHED --2025-10-08 14:39:10--
Total wall clock time: 0.3s
Downloaded: 1 files, 66K in 0.06s (1.11 MB/s)
Archive:  /content/SpeedTransformer.zip
   creating: SpeedTransformer/
  inflating: SpeedTransformer/.DS_Store  
  inflating: __MACOSX/SpeedTransformer/._.DS_Store  
   creating: SpeedTransformer/models/
  inflating: __MACOSX/SpeedTransformer/._models  
   creating: SpeedTransformer/data/
  inflating: SpeedTran

In [ ]:
# Download CSV files into the data/ directory
!wget -O /content/SpeedTransformer/data/mobis_processed.csv https://zenodo.org/record/17290562/files/mobis_processed.csv
!wget -O /content/SpeedTransformer/data/geolife_processed.csv https://zenodo.org/record/17290562/files/geolife_processed.csv
!wget -O /content/SpeedTransformer/data/miniprogram_balanced.csv https://zenodo.org/record/17290562/files/miniprogram_balanced.csv

--2025-10-08 14:39:10--  https://zenodo.org/record/17290562/files/mobis_processed.csv
Resolving zenodo.org (zenodo.org)... 188.185.43.25, 188.185.48.194, 188.185.45.92, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.25|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/17290562/files/mobis_processed.csv [following]
--2025-10-08 14:39:11--  https://zenodo.org/records/17290562/files/mobis_processed.csv
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 31273348118 (29G) [text/plain]
Saving to: ‘/content/SpeedTransformer/data/mobis_processed.csv’

/content/SpeedTrans 100%[===================>]  29.12G   149MB/s    in 37m 12s 

2025-10-08 15:16:23 (13.4 MB/s) - ‘/content/SpeedTransformer/data/mobis_processed.csv’ saved [31273348118/31273348118]

--2025-10-08 15:16:23--  https://zenodo.org/record/17290562/files/geolife_processed.csv
Resolving zenodo.org (zenodo.org)... 188.185.45.92, 18

In [ ]:
!mv /content/SpeedTransformer /content/A-SpeedTransformer

## Training Benchmark Runs: LSTM and SpeedTransformer on Geolife & MOBIS


Trains SpeedTransformer and the LSTM baseline from scratch on both MOBIS and Geolife. It rebuilds the headline benchmark comparisons—overall accuracy, convergence curves, and class-level F1 scores—used in Section 5 “Experiments” (see Table 5 and Figures 2–3). These outputs show how the transformer surpasses the LSTM when both see the full datasets with identical preprocessing.

In [ ]:
%cd /content/
!bash A-SpeedTransformer/models/replication/run_training_experiments.sh

/content
\n[Transformer] Geolife → lr2e-4_bs512_h8_d128_kv4_do0.1
2025-10-08 15:25:29,139 - INFO - Setting random seed to 316
2025-10-08 15:25:29,139 - INFO - DDP world_size=1 local_rank=0
2025-10-08 15:25:29,140 - INFO - Initializing DataProcessor...
Extracting unique trajectory IDs...
Reading traj_ids: 5it [00:02,  2.26it/s]
Total unique traj_ids found: 9347
Splitting traj_ids into train, validation, and test sets...
Train: 6542, Val: 1402, Test: 1403
Fitting label encoder on target column...
Reading labels for encoding: 5it [00:02,  2.43it/s]
Classes found: ['bike' 'bus' 'car' 'train' 'walk']
Fitting scaler on training features...
Reading training features for scaling: 5it [00:02,  1.95it/s]
Scaler fitting completed.
Creating sequences in sliding windows...
Processing chunks: 5it [00:10,  2.05s/it]
Train: 51515, Val: 10762, Test: 10729
Skipped sequences (zero length): 0
Scaler saved to /content/A-SpeedTransformer/models/transformer/experiments/geolife_transformer_sweeps/lr2e-4_bs512

## Geolife Window Sweep

Repeats the Geolife sliding-window size sweep, holding hyperparameters fixed while varying the context length fed to the transformer. It directly supports the window-size discussion in Appendix F/Section \ref{window_size}, reproducing Figure 16 and Table 16 that justify the T = 200 window used throughout the paper.

In [ ]:
!bash A-SpeedTransformer/models/replication/run_window_sweep_experiments.sh

[Window Sweep] Geolife → geolife_ws200_lr2e-4_bs512_h8_d128_kv4_do0.1
2025-10-09 01:18:28,886 - INFO - Setting random seed to 316
2025-10-09 01:18:28,886 - INFO - DDP world_size=1 local_rank=0
2025-10-09 01:18:28,887 - INFO - Initializing DataProcessor...
Extracting unique trajectory IDs...
Reading traj_ids: 5it [00:02,  2.26it/s]
Total unique traj_ids found: 9347
Splitting traj_ids into train, validation, and test sets...
Train: 6542, Val: 1402, Test: 1403
Fitting label encoder on target column...
Reading labels for encoding: 5it [00:02,  2.43it/s]
Classes found: ['bike' 'bus' 'car' 'train' 'walk']
Fitting scaler on training features...
Reading training features for scaling: 5it [00:02,  1.96it/s]
Scaler fitting completed.
Creating sequences in sliding windows...
Processing chunks: 5it [00:10,  2.06s/it]
Train: 51515, Val: 10762, Test: 10729
Skipped sequences (zero length): 0
Scaler saved to /content/A-SpeedTransformer/models/transformer/experiments/geolife_window_sweeps/geolife_ws200

## Full Geolife Finetuning



Starts from MOBIS-pretrained checkpoints and fine-tunes on 100 and 200 labeled Geolife trips. The resulting checkpoints and logs feed the low-shot transfer analysis in Section 5.2 (Table 6), demonstrating that SpeedTransformer retains strong accuracy with only 1–2 % of the Geolife training trajectories.

In [ ]:
!bash A-SpeedTransformer/models/replication/run_gl_finetune_experiments.sh

\n[Transformer Finetune] Geolife → lr2e-4_warmup0_freeze_attention
2025-10-09 01:28:12,398 - INFO - Initializing DataProcessor for fine-tuning...
Extracting unique trajectory IDs...
Reading traj_ids: 5it [00:02,  2.16it/s]
Total unique traj_ids found: 9347
Splitting traj_ids into train, validation, and test sets...
Train: 93, Val: 1869, Test: 7385
Fitting label encoder on target column...
Reading labels for encoding: 5it [00:02,  2.33it/s]
Classes found: ['bike' 'bus' 'car' 'train' 'walk']
Fitting scaler on training features...
Reading training features for scaling: 5it [00:02,  1.87it/s]
Scaler fitting completed.
Creating sequences in sliding windows...
Processing chunks: 5it [00:10,  2.06s/it]
Train: 899, Val: 16344, Test: 55763
Skipped sequences (zero length): 0
Creating sequences in sliding windows...
Processing chunks: 5it [00:10,  2.02s/it]
Train: 899, Val: 16344, Test: 55763
Skipped sequences (zero length): 0
/content/A-SpeedTransformer/models/transformer/model_utils.py:231: Fut

## Low-Shot Geolife Adaptation

Starts from MOBIS-pretrained checkpoints and fine-tunes on 100 and 200 labeled Geolife trips. The resulting checkpoints and logs feed the low-shot transfer analysis in Section 5.2 (Table 6), demonstrating that SpeedTransformer retains strong accuracy with only 1–2 % of the Geolife training trajectories.

In [ ]:
!bash A-SpeedTransformer/models/replication/run_gl_lowshot_finetune_experiments.sh

\n[lowshot] Target train trajs: 100
  val_size=0.494597196961592, test_size=0.494704183160372
  transformer run=train100_lr2e-4_warmup0_freeze_attention
  transformer saved → /content/A-SpeedTransformer/models/transformer/experiments/finetune_lowshot/train100_lr2e-4_warmup0_freeze_attention
  lstm run=train100_lr5e-5_bs128_do0.3
  lstm saved → /content/A-SpeedTransformer/models/lstm/experiments/finetune_lowshot/train100_lr5e-5_bs128_do0.3
\n[lowshot] Target train trajs: 200
  val_size=0.489247887022574, test_size=0.489354873221354
  transformer run=train200_lr2e-4_warmup0_freeze_attention
  transformer saved → /content/A-SpeedTransformer/models/transformer/experiments/finetune_lowshot/train200_lr2e-4_warmup0_freeze_attention
  lstm run=train200_lr5e-5_bs128_do0.3
  lstm saved → /content/A-SpeedTransformer/models/lstm/experiments/finetune_lowshot/train200_lr5e-5_bs128_do0.3
\nLow-shot Geolife finetuning runs completed.


## CarbonClever Field Study Finetuning


Adapts both architectures to the real-world CarbonClever dataset collected via the WeChat mini-app. Its checkpoints and logs recreate Table 7 in Section 6 “Real-World Validation,” confirming SpeedTransformer’s robustness when confronted with noisy, heterogeneous smartphone GPS traces.

In [ ]:
!bash A-SpeedTransformer/models/replication/run_miniprogram_finetune_experiments.sh

Missing LSTM pretrained weights at /content/A-SpeedTransformer/models/lstm/experiments/mobis_lstm_sweeps/mobis_lr1e-3_bs128_h128_l2_do0.1. Run the original MOBIS LSTM sweep to generate them.


## Replicate figures

Consumes the artifacts from the five pipelines above—logs, metrics CSVs, best-model checkpoints—and regenerates every figure and the experiment summary table cited in the paper (Figures 2–7, 15–16, Table 5). Running it ensures the plots in the manuscript align exactly with the Colab notebook outputs.

In [ ]:
!python /content/A-SpeedTransformer/models/replication/metrics_gen.py

=== ANALYZING ALL EXPERIMENTS ===

Found experiments:
  Geolife: 1 Transformer, 1 LSTM
  Mobis: 1 Transformer, 1 LSTM
  Finetune Sweeps: 0 Transformer, 0 LSTM
  Miniprogram: 0 Transformer, 0 LSTM
  Geolife Window Sweeps: 1 Transformer
  Geolife Low-shot Finetune: 2 Transformer
Target params: {'lr': '2e-4', 'bs': 512, 'dropout': 0.1}
Target params: {'lr': '1e-4', 'bs': 512, 'dropout': 0.1}

=== FINDING BEST MATCHES FOR ORIGINAL EXPERIMENTS ===
Best Geolife Transformer: lr2e-4_bs512_h8_d128_kv4_do0.1 (acc: 0.9576)
Matching Geolife LSTM: geolife_lr5e-4_bs128_h256_l3_do0.1 (acc: 0.9273)
Best Mobis Transformer: mobis_lr1e-4_bs512_h8_d128_kv4_do0.1 (acc: 0.9427)
Matching Mobis LSTM: mobis_lr5e-4_bs128_h256_l3_do0.1 (acc: 0.9211)

=== BEST GEOLIFE TRANSFORMER RESULTS ===
1. lr2e-4_bs512_h8_d128_kv4_do0.1: 0.9576

=== BEST GEOLIFE LSTM RESULTS ===
1. geolife_lr5e-4_bs128_h256_l3_do0.1: 0.9273

=== BEST MOBIS TRANSFORMER RESULTS ===
1. mobis_lr1e-4_bs512_h8_d128_kv4_do0.1: 0.9427

=== BEST MOBI